# Faster-Whisper ASR Training for Twi/Akan - WITH MANIFESTS
## High-Performance OpenAI Whisper Fine-tuning using Enhanced Manifests

This notebook uses the standardized manifests generated by the Enhanced Multi-Rate Data Processing Pipeline.

### Key Features:
- **Uses ASR 16kHz Manifests**: Optimized for Whisper's 16kHz requirement
- **Quality-Filtered Data**: Pre-processed and normalized audio
- **Simplified Pipeline**: Focus on training, not data preprocessing
- **CTranslate2 Optimization**: Production-ready deployment
- **Multi-Dialect Support**: Balanced Twi dialect representation

### Expected Performance:
- **WER <15%**: Improved accuracy through quality data
- **5-10x Speed**: CTranslate2 optimization
- **Real-time Inference**: Production deployment ready

In [ ]:
# Installation and setup
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers datasets accelerate evaluate
!pip install -q faster-whisper openai-whisper
!pip install -q jiwer librosa soundfile tensorboard
!pip install -q optimum ctranslate2
!apt-get install -qq ffmpeg

import os
import sys
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional, Union
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

# Whisper and audio processing
import whisper
from faster_whisper import WhisperModel
import librosa
import soundfile as sf
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from transformers import TrainingArguments, Trainer
from datasets import Dataset, DatasetDict, Audio
import evaluate

print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🎯 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Environment detection and configuration
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔍 Detected Google Colab environment")
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = Path('/content/drive/MyDrive/voice-lab-data')
else:
    print("🔍 Detected local environment")
    DATA_PATH = Path('/opt/kasaai/apps/voice-lab/data')

print(f"📂 Data path: {DATA_PATH}")

@dataclass
class WhisperManifestConfig:
    """Configuration for Whisper training using manifests."""
    # Model settings
    model_name: str = "openai/whisper-small"
    language: str = "twi"  # Target language
    task: str = "transcribe"
    
    # Data paths
    data_dir: Path = DATA_PATH
    manifest_dir: Path = DATA_PATH / "processed/model_manifests"
    output_dir: Path = DATA_PATH / "models/faster_whisper_twi_manifests"
    ctranslate2_dir: Path = DATA_PATH / "models/faster_whisper_twi_ct2"
    
    # Training parameters
    batch_size: int = 16
    gradient_accumulation_steps: int = 2
    learning_rate: float = 1e-5
    warmup_steps: int = 500
    num_train_epochs: int = 5
    fp16: bool = True
    
    # Audio settings (matches enhanced processing)
    sample_rate: int = 16000
    max_audio_length: int = 30  # seconds
    
    # Evaluation
    eval_steps: int = 500
    save_steps: int = 1000
    logging_steps: int = 100
    
    # CTranslate2 optimization
    ct2_quantization: str = "int8"  # Options: int8, float16, float32
    
    def __post_init__(self):
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.ctranslate2_dir.mkdir(parents=True, exist_ok=True)

config = WhisperManifestConfig()
print(f"📋 Whisper Manifest Configuration loaded")
print(f"   Model: {config.model_name}")
print(f"   Language: {config.language}")
print(f"   Manifest directory: {config.manifest_dir}")
print(f"   Output directory: {config.output_dir}")

In [ ]:
# Enhanced manifest loader for Whisper ASR
class WhisperManifestLoader:
    """Load ASR manifests optimized for Whisper training."""
    
    def __init__(self, config: WhisperManifestConfig):
        self.config = config
        
    def load_whisper_manifests(self) -> Dict[str, pd.DataFrame]:
        """Load Whisper training manifests from enhanced processing pipeline."""
        print("📋 Loading Whisper ASR manifests...")
        
        manifests = {}
        
        # Primary manifest files for Whisper (16kHz)
        whisper_manifest_files = {
            'train': self.config.manifest_dir / 'faster_whisper_train.csv',
            'val': self.config.manifest_dir / 'faster_whisper_val.csv'
        }
        
        # Fallback to other ASR manifests
        fallback_manifests = {
            'train': [
                self.config.manifest_dir / 'wav2vec2_train.csv',
                self.config.manifest_dir / 'mms_train.csv',
                self.config.data_dir / 'processed/manifests/asr_16k_dataset.csv'
            ],
            'val': [
                self.config.manifest_dir / 'wav2vec2_val.csv',
                self.config.manifest_dir / 'mms_val.csv',
                self.config.data_dir / 'processed/manifests/asr_16k_dataset.csv'
            ]
        }
        
        for split_name in ['train', 'val']:
            manifest_path = whisper_manifest_files[split_name]
            
            if manifest_path.exists():
                print(f"   ✅ Found Whisper manifest: {manifest_path.name}")
                try:
                    df = pd.read_csv(manifest_path)
                    manifests[split_name] = self._validate_whisper_manifest(df, split_name)
                except Exception as e:
                    print(f"   ⚠️ Error loading {manifest_path}: {e}")
                    manifests[split_name] = pd.DataFrame()
            else:
                print(f"   ⚠️ Whisper manifest not found: {manifest_path.name}")
                print(f"   🔄 Looking for fallback manifests...")
                
                # Try fallback manifests
                fallback_df = self._load_fallback_manifests(fallback_manifests[split_name])
                
                if not fallback_df.empty:
                    # Split fallback data if it's a single dataset file
                    if split_name == 'val' and len(fallback_manifests['val']) > 0:
                        train_size = int(0.9 * len(fallback_df))
                        if split_name == 'train':
                            fallback_df = fallback_df.iloc[:train_size]
                        else:
                            fallback_df = fallback_df.iloc[train_size:]
                    
                    manifests[split_name] = self._validate_whisper_manifest(fallback_df, split_name)
                else:
                    print(f"   ❌ No valid manifests found for {split_name}")
                    manifests[split_name] = self._create_dummy_whisper_manifest()
        
        return manifests
    
    def _load_fallback_manifests(self, fallback_paths: List[Path]) -> pd.DataFrame:
        """Load fallback manifests and combine them."""
        combined_df = pd.DataFrame()
        
        for fallback_path in fallback_paths:
            if fallback_path.exists():
                print(f"   ✅ Using fallback: {fallback_path.name}")
                try:
                    df = pd.read_csv(fallback_path)
                    combined_df = pd.concat([combined_df, df], ignore_index=True)
                except Exception as e:
                    print(f"   ⚠️ Error loading {fallback_path}: {e}")
                    continue
        
        return combined_df
    
    def _validate_whisper_manifest(self, df: pd.DataFrame, split_name: str) -> pd.DataFrame:
        """Validate and format manifest for Whisper training."""
        print(f"   🔍 Validating {split_name} manifest for Whisper...")
        
        if df.empty:
            return df
        
        # Required columns for Whisper ASR
        required_columns = ['audio_path', 'transcript']
        
        # Map common column variations
        column_mappings = {
            'text': 'transcript',
            'transcription': 'transcript',
            'sentence': 'transcript',
            'audio': 'audio_path',
            'path': 'audio_path',
            'file_path': 'audio_path',
            'processed_audio_path': 'audio_path'
        }
        
        for old_col, new_col in column_mappings.items():
            if old_col in df.columns and new_col not in df.columns:
                df = df.rename(columns={old_col: new_col})
                print(f"   🔄 Mapped {old_col} → {new_col}")
        
        # Check required columns
        missing_columns = set(required_columns) - set(df.columns)
        if missing_columns:
            print(f"   ❌ Missing required columns: {missing_columns}")
            return pd.DataFrame()
        
        # Add optional columns
        if 'duration' not in df.columns:
            df['duration'] = 5.0  # Default duration
        if 'language' not in df.columns:
            df['language'] = 'tw'
        if 'speaker_id' not in df.columns:
            df['speaker_id'] = 'unknown'
        
        # Quality filtering
        initial_count = len(df)
        
        # Filter valid audio files
        df['audio_exists'] = df['audio_path'].apply(lambda x: Path(x).exists() if pd.notna(x) else False)
        df = df[df['audio_exists']].drop(columns=['audio_exists'])
        
        # Filter valid transcripts
        df = df[df['transcript'].str.len() > 0]
        
        # Filter audio duration (Whisper works best with <30s)
        if 'duration' in df.columns:
            df = df[df['duration'] <= self.config.max_audio_length]
        
        final_count = len(df)
        retention_rate = (final_count / initial_count * 100) if initial_count > 0 else 0
        
        print(f"   📊 {split_name}: {final_count}/{initial_count} samples retained ({retention_rate:.1f}%)")
        
        return df
    
    def _create_dummy_whisper_manifest(self) -> pd.DataFrame:
        """Create dummy manifest for Whisper testing."""
        print("   📝 Creating dummy Whisper manifest...")
        
        dummy_dir = self.config.data_dir / 'dummy_audio_whisper'
        dummy_dir.mkdir(parents=True, exist_ok=True)
        
        samples = [
            "Wo ho te sɛn? Me ho yɛ.",
            "Me ma wo akye.",
            "Nante yie.",
            "Me da wo ase.",
            "Ɛyɛ dɛn?",
            "Mepɛ aduane.",
            "Ɛyɛ anɔpa anum.",
            "Me kɔ sukuu."
        ]
        
        dummy_data = []
        for i, text in enumerate(samples):
            audio_file = dummy_dir / f"whisper_dummy_{i:03d}.wav"
            
            if not audio_file.exists():
                # Create varied duration audio (2-8 seconds)
                duration = np.random.uniform(2.0, 8.0)
                dummy_audio = np.random.normal(0, 0.1, int(16000 * duration)).astype(np.float32)
                sf.write(audio_file, dummy_audio, 16000)
            
            dummy_data.append({
                'audio_path': str(audio_file),
                'transcript': text,
                'duration': sf.info(str(audio_file)).duration,
                'language': 'tw',
                'speaker_id': f'whisper_dummy_{i % 3}',
                'dataset': 'dummy_whisper'
            })
        
        return pd.DataFrame(dummy_data)

# Load manifests
manifest_loader = WhisperManifestLoader(config)
manifests = manifest_loader.load_whisper_manifests()

print(f"\n📊 Whisper Manifest Summary:")
for split_name, df in manifests.items():
    print(f"   {split_name}: {len(df)} samples")
    if not df.empty:
        print(f"     Unique speakers: {df['speaker_id'].nunique()}")
        print(f"     Total duration: {df['duration'].sum()/60:.1f} minutes")
        print(f"     Average duration: {df['duration'].mean():.1f}s")
        print(f"     Languages: {df['language'].unique()}")

In [ ]:
# Initialize Whisper model and processor
class WhisperManifestTrainer:
    """Whisper trainer using manifest-based data loading."""
    
    def __init__(self, config: WhisperManifestConfig):
        self.config = config
        self.model = None
        self.processor = None
        
    def load_model_and_processor(self):
        """Load Whisper model and processor."""
        print(f"🚀 Loading Whisper model: {self.config.model_name}")
        
        # Load processor
        self.processor = WhisperProcessor.from_pretrained(self.config.model_name)
        
        # Load model
        self.model = WhisperForConditionalGeneration.from_pretrained(self.config.model_name)
        
        # Configure for Twi
        self.model.config.forced_decoder_ids = None
        self.model.config.suppress_tokens = []
        
        # Try to set language token (Twi may not be officially supported)
        try:
            # Force the model to transcribe in English initially, then we'll fine-tune for Twi
            self.model.generation_config.language = "<|en|>"
            self.model.generation_config.task = "transcribe"
            print(f"   ✅ Model configured for transcription")
        except Exception as e:
            print(f"   ⚠️ Language configuration warning: {e}")
        
        print(f"\n📊 Model Summary:")
        print(f"   Parameters: {self.model.num_parameters():,}")
        print(f"   Model size: {self.config.model_name.split('-')[-1]}")
        
        return self.model, self.processor
        
    def prepare_dataset(self, manifests: Dict[str, pd.DataFrame]) -> DatasetDict:
        """Convert manifests to Whisper datasets."""
        print("🔄 Preparing Whisper datasets from manifests...")
        
        datasets = {}
        
        for split_name, df in manifests.items():
            if df.empty:
                continue
            
            print(f"   📊 Processing {split_name}: {len(df)} samples")
            
            # Convert to HuggingFace dataset
            dataset = Dataset.from_pandas(df)
            
            # Add audio column
            dataset = dataset.cast_column("audio_path", Audio(sampling_rate=self.config.sample_rate))
            dataset = dataset.rename_column("audio_path", "audio")
            
            datasets[split_name] = dataset
        
        # Ensure train/validation split
        if 'train' in datasets and 'val' not in datasets:
            print("   🔄 Creating validation split from training data")
            train_val_split = datasets['train'].train_test_split(test_size=0.1, seed=42)
            datasets['train'] = train_val_split['train']
            datasets['val'] = train_val_split['test']
        
        dataset_dict = DatasetDict(datasets)
        
        # Apply preprocessing
        dataset_dict = dataset_dict.map(
            self._prepare_batch,
            batched=True,
            batch_size=8,
            remove_columns=dataset_dict["train"].column_names
        )
        
        print(f"\n✅ Dataset preparation complete:")
        for split_name, dataset in dataset_dict.items():
            print(f"   {split_name}: {len(dataset)} samples")
        
        return dataset_dict
    
    def _prepare_batch(self, batch):
        """Prepare batch for Whisper training."""
        # Process audio
        audio_arrays = [x["array"] for x in batch["audio"]]
        
        # Process audio features
        inputs = self.processor.feature_extractor(
            audio_arrays,
            sampling_rate=self.config.sample_rate,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=3000  # ~30 seconds at 16kHz
        )
        
        # Process text labels
        labels = self.processor.tokenizer(
            batch["transcript"],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=448  # Whisper's max text length
        )
        
        return {
            "input_features": inputs.input_features,
            "labels": labels.input_ids
        }

# Initialize trainer
whisper_trainer = WhisperManifestTrainer(config)
model, processor = whisper_trainer.load_model_and_processor()

# Prepare datasets if manifests are available
if manifests['train'] is not None and not manifests['train'].empty:
    datasets = whisper_trainer.prepare_dataset(manifests)
    print("✅ Whisper datasets prepared successfully!")
else:
    print("❌ No training manifests available")
    datasets = None

In [ ]:
# Data collator for Whisper
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class WhisperDataCollatorWithPadding:
    """Data collator for Whisper training."""
    
    processor: Any
    decoder_start_token_id: int = None
    
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Split inputs and labels
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        
        # Pad inputs
        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )
        
        # Pad labels
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )
        
        # Replace padding with -100 for loss computation
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        
        # Add decoder start token if needed
        if self.decoder_start_token_id is not None:
            labels[:, 0] = self.decoder_start_token_id
        
        batch["labels"] = labels
        
        return batch

# Initialize data collator
data_collator = WhisperDataCollatorWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id
)

print("✅ Data collator initialized")

In [ ]:
# Training setup and metrics
def compute_metrics(eval_pred):
    """Compute WER and BLEU metrics for Whisper."""
    pred_ids = eval_pred.predictions
    label_ids = eval_pred.label_ids
    
    # Replace -100 with pad token
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    
    # Decode predictions and labels
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    
    # Compute WER
    wer_metric = evaluate.load("wer")
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    
    # Additional metrics
    exact_match = sum([p.strip().lower() == r.strip().lower() for p, r in zip(pred_str, label_str)]) / len(pred_str)
    
    return {
        "wer": wer,
        "exact_match": exact_match
    }

if datasets is not None:
    print("🏋️ Setting up Whisper training...")
    
    training_args = TrainingArguments(
        output_dir=str(config.output_dir),
        per_device_train_batch_size=config.batch_size,
        per_device_eval_batch_size=config.batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        learning_rate=config.learning_rate,
        warmup_steps=config.warmup_steps,
        num_train_epochs=config.num_train_epochs,
        fp16=config.fp16,
        eval_strategy="steps",
        eval_steps=config.eval_steps,
        save_strategy="steps",
        save_steps=config.save_steps,
        logging_steps=config.logging_steps,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        save_total_limit=3,
        push_to_hub=False,
        remove_unused_columns=False,
        dataloader_num_workers=2
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=datasets["train"],
        eval_dataset=datasets["val"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        tokenizer=processor.feature_extractor
    )
    
    print("\n✅ Whisper training setup complete!")
    print(f"   Training samples: {len(datasets['train'])}")
    print(f"   Validation samples: {len(datasets['val'])}")
    print(f"   Batch size: {config.batch_size}")
    print(f"   Learning rate: {config.learning_rate}")
    print(f"   Epochs: {config.num_train_epochs}")
    
    print("\n🚀 Ready to start training! Uncomment the next lines to begin:")
    print("# train_result = trainer.train()")
    print("# trainer.save_model()")
    print("# processor.save_pretrained(config.output_dir)")
    
    # Uncomment to start training
    # train_result = trainer.train()
    # trainer.save_model()
    # processor.save_pretrained(config.output_dir)
    # print(f"✅ Training complete! Model saved to {config.output_dir}")
    
else:
    print("❌ Cannot setup training - no datasets available")

In [ ]:
# CTranslate2 conversion and optimization
def convert_to_ctranslate2():
    """Convert trained Whisper model to CTranslate2 format."""
    print("🔄 Converting Whisper model to CTranslate2 format...")
    
    if not config.output_dir.exists():
        print(f"   ❌ Trained model not found at {config.output_dir}")
        print("   Please train the model first or update the path.")
        return None
    
    try:
        import ct2_model
        
        # Convert model
        ct2_model.convert_transformers(
            str(config.output_dir),
            str(config.ctranslate2_dir),
            quantization=config.ct2_quantization,
            force=True
        )
        
        print(f"   ✅ Model converted to CTranslate2: {config.ctranslate2_dir}")
        print(f"   Quantization: {config.ct2_quantization}")
        
        return config.ctranslate2_dir
        
    except Exception as e:
        print(f"   ❌ Conversion failed: {e}")
        
        # Alternative: Use command line conversion
        print("   🔄 Trying alternative conversion method...")
        
        cmd = f"""ct2-transformers-converter --model {config.output_dir} \
                 --output_dir {config.ctranslate2_dir} \
                 --quantization {config.ct2_quantization} \
                 --force"""
        
        print(f"   Run this command manually:")
        print(f"   {cmd}")
        
        return None

def test_faster_whisper():
    """Test the converted Faster-Whisper model."""
    print("🧪 Testing Faster-Whisper model...")
    
    # Check if CTranslate2 model exists
    if not config.ctranslate2_dir.exists():
        print(f"   ⚠️ CTranslate2 model not found at {config.ctranslate2_dir}")
        print("   Please convert the model first.")
        return
    
    try:
        # Load Faster-Whisper model
        model_fw = WhisperModel(str(config.ctranslate2_dir), device="cuda" if torch.cuda.is_available() else "cpu")
        
        # Test with sample audio
        if datasets and 'val' in datasets and len(datasets['val']) > 0:
            test_sample = datasets['val'][0]
            audio_array = test_sample['input_features']
            
            # Convert to numpy if tensor
            if isinstance(audio_array, torch.Tensor):
                audio_array = audio_array.numpy()
            
            # Transcribe
            import time
            start_time = time.time()
            
            segments, info = model_fw.transcribe(
                audio_array,
                language="en",  # Fallback to English for now
                task="transcribe"
            )
            
            inference_time = time.time() - start_time
            
            # Print results
            transcription = " ".join([segment.text for segment in segments])
            
            print(f"\n📊 Faster-Whisper Test Results:")
            print(f"   Transcription: {transcription}")
            print(f"   Inference time: {inference_time:.3f}s")
            print(f"   Language detected: {info.language}")
            print(f"   Language probability: {info.language_probability:.3f}")
            
        else:
            print("   ⚠️ No test data available")
        
        print("\n✅ Faster-Whisper model working correctly!")
        
    except Exception as e:
        print(f"   ❌ Faster-Whisper test failed: {e}")

def generate_deployment_examples():
    """Generate deployment examples."""
    print("\n📚 Deployment Examples:")
    
    # Faster-Whisper usage
    faster_whisper_code = f'''
# Faster-Whisper (Production Deployment)
from faster_whisper import WhisperModel
import librosa

# Load optimized model
model = WhisperModel(
    "{config.ctranslate2_dir}",
    device="cuda",  # or "cpu"
    compute_type="{config.ct2_quantization}"
)

# Transcribe audio file
audio, sr = librosa.load("twi_audio.wav", sr=16000)
segments, info = model.transcribe(
    audio,
    language="en",  # Use English as base, trained for Twi
    task="transcribe"
)

transcription = " ".join([segment.text for segment in segments])
print(f"Transcription: {{transcription}}")
'''
    
    # Original Whisper usage
    whisper_code = f'''
# Original Whisper (Higher Accuracy)
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch
import librosa

# Load fine-tuned model
processor = WhisperProcessor.from_pretrained("{config.output_dir}")
model = WhisperForConditionalGeneration.from_pretrained("{config.output_dir}")

# Transcribe audio
audio, sr = librosa.load("twi_audio.wav", sr=16000)
inputs = processor(audio, sampling_rate=16000, return_tensors="pt")

with torch.no_grad():
    predicted_ids = model.generate(**inputs)

transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
print(f"Transcription: {{transcription}}")
'''
    
    print("### Faster-Whisper (5-10x Speed):")
    print(faster_whisper_code)
    
    print("\n### Original Whisper (Higher Accuracy):")
    print(whisper_code)

print("\n🎯 CTranslate2 and Testing Functions Ready:")
print("# ct2_path = convert_to_ctranslate2()")
print("# test_faster_whisper()")
print("# generate_deployment_examples()")

## Summary: Faster-Whisper Training with Manifests

This notebook demonstrates the complete pipeline for training and optimizing Whisper models using the Enhanced Multi-Rate Data Processing manifests.

### ✅ Key Features:

1. **Manifest-Based Loading**
   - Uses standardized 16kHz ASR manifests
   - Automatic fallback to compatible manifests
   - Quality-filtered and normalized audio data

2. **Optimized Training Pipeline**
   - Fine-tuned for Twi language
   - Proper audio preprocessing (16kHz, <30s clips)
   - WER and exact match metrics

3. **Production Optimization**
   - CTranslate2 conversion for speed
   - INT8 quantization options
   - Faster-Whisper integration

4. **Deployment Ready**
   - Both accuracy and speed variants
   - Complete usage examples
   - Performance benchmarking

### 🚀 Usage Workflow:

1. **Training**: `trainer.train()` using manifest data
2. **Optimization**: `convert_to_ctranslate2()` for speed
3. **Testing**: `test_faster_whisper()` for validation
4. **Deployment**: Use provided examples

### 📊 Expected Performance:

- **Original Model**: Higher accuracy, slower inference
- **Faster-Whisper**: 5-10x speed improvement with minimal accuracy loss
- **Target WER**: <15% on Twi speech
- **Real-time Factor**: <0.1x (faster than real-time)

This approach provides a complete solution from training to production deployment, leveraging the standardized data processing pipeline for consistent, high-quality results.